# Module 5: EEG Basics -- Reading the Brain

This tutorial walks through the concepts step by step. Each section includes an explanation, an example, and an exercise.

## Step 1: Setup & Imports

This notebook covers the fundamentals of EEG (electroencephalography): where electrodes go on the scalp (the **10-20 system**), what the five major **frequency bands** mean clinically, and the key phenomenon of **alpha blocking**.

EEG records the summed postsynaptic potentials of millions of cortical pyramidal neurons through scalp electrodes. The signal is measured in microvolts, is inherently noisy, and carries clinically vital information.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
%matplotlib inline

# Plotting defaults
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

**Exercise 1:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 2: The 10-20 System: Where Electrodes Go

The **International 10-20 System** is the standard electrode placement scheme used worldwide. Electrodes are spaced at 10% or 20% intervals along the skull from standard landmarks (nasion, inion, preauricular points).

**Naming convention:**
- Letter = brain region (Fp=frontopolar, F=frontal, C=central, T=temporal, P=parietal, O=occipital)
- **Odd numbers = left hemisphere**, **even numbers = right hemisphere**, **z = midline**
- So C3 = left central, C4 = right central, Cz = midline central

This system is universal -- the same in a hospital in Tokyo, Toronto, or Tunis.

In [ ]:
# Draw a 10-20 system head map
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.set_aspect('equal')
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.4)
ax.axis('off')
ax.set_title('International 10-20 Electrode System', fontsize=16, fontweight='bold')

# Head outline
theta = np.linspace(0, 2*np.pi, 100)
ax.plot(np.cos(theta), np.sin(theta), 'k-', linewidth=2)

# Nose
ax.plot([-.08, 0, .08], [1.0, 1.15, 1.0], 'k-', linewidth=2)
ax.text(0, 1.25, 'Nose (anterior)', ha='center', fontsize=10, style='italic', color='gray')

# Ears
for side in [-1, 1]:
    ear_x = side * 1.05
    ax.plot([ear_x, ear_x+side*0.05, ear_x+side*0.05, ear_x],
            [-0.1, -0.1, 0.1, 0.1], 'k-', linewidth=1.5)

# Electrode positions (approximate 10-20 layout on unit circle)
electrodes = {
    # Frontal (blue)
    'Fp1': (-0.25, 0.85), 'Fp2': (0.25, 0.85),
    'F7': (-0.7, 0.5), 'F3': (-0.35, 0.55), 'Fz': (0, 0.55),
    'F4': (0.35, 0.55), 'F8': (0.7, 0.5),
    # Central (green)
    'T3': (-0.9, 0), 'C3': (-0.4, 0), 'Cz': (0, 0), 
    'C4': (0.4, 0), 'T4': (0.9, 0),
    # Parietal / Temporal (orange)
    'T5': (-0.7, -0.45), 'P3': (-0.35, -0.45), 'Pz': (0, -0.45),
    'P4': (0.35, -0.45), 'T6': (0.7, -0.45),
    # Occipital (red)
    'O1': (-0.25, -0.8), 'O2': (0.25, -0.8),
}

# Color by region
region_colors = {
    'Fp': '#3498db', 'F': '#3498db', 
    'C': '#27ae60', 'T': '#f39c12',
    'P': '#e67e22', 'O': '#e74c3c'
}

def get_color(name):
    for prefix in ['Fp', 'F', 'C', 'T', 'P', 'O']:
        if name.startswith(prefix):
            return region_colors[prefix]
    return 'gray'

for name, (x, y) in electrodes.items():
    color = get_color(name)
    circle = plt.Circle((x, y), 0.08, color=color, ec='white', linewidth=2, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, name, ha='center', va='center', fontsize=7, 
            fontweight='bold', color='white', zorder=4)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', label='Frontal'),
    Patch(facecolor='#27ae60', label='Central'),
    Patch(facecolor='#f39c12', label='Temporal'),
    Patch(facecolor='#e67e22', label='Parietal'),
    Patch(facecolor='#e74c3c', label='Occipital'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

**Exercise 2:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 3: EEG Frequency Bands: The Language of Brain Rhythms

EEG activity is categorized into five major frequency bands. Each has a name, frequency range, and clinical meaning. This is the vocabulary every neurologist uses.

| Band | Range | Amplitude | Best Location | Clinical Significance |
|------|-------|-----------|---------------|----------------------|
| **Delta** | 1-4 Hz | 100-200 uV | Diffuse | Normal in deep sleep. Pathological if awake (encephalopathy). |
| **Theta** | 4-8 Hz | 50-100 uV | Temporal | Normal in drowsiness/children. Focal = possible lesion. |
| **Alpha** | 8-13 Hz | 30-50 uV | Occipital | Relaxed, eyes closed. Blocks with eye opening. |
| **Beta** | 13-30 Hz | 10-20 uV | Frontal/Central | Alert state, anxiety, benzodiazepines. |
| **Gamma** | >30 Hz | 5-10 uV | Variable | Cognitive processing. Often contaminated by muscle artifact. |

Notice the inverse relationship: **slower rhythms have higher amplitude**, faster rhythms are tiny.

In [ ]:
# Generate and plot signals in each EEG band
np.random.seed(42)
fs = 512  # Sampling rate
duration = 2  # seconds
t = np.arange(0, duration, 1/fs)

# Define the five EEG bands
bands = {
    'Delta (1-4 Hz)':   {'freq': 2.5,  'amp': 150, 'color': '#2c3e50'},
    'Theta (4-8 Hz)':   {'freq': 6,    'amp': 75,  'color': '#1abc9c'},
    'Alpha (8-13 Hz)':  {'freq': 10,   'amp': 40,  'color': '#8e44ad'},
    'Beta (13-30 Hz)':  {'freq': 20,   'amp': 15,  'color': '#e67e22'},
    'Gamma (>30 Hz)':   {'freq': 40,   'amp': 7,   'color': '#e74c3c'},
}

fig, axes = plt.subplots(5, 1, figsize=(12, 10), sharex=True)
fig.suptitle('EEG Frequency Bands', fontsize=16, fontweight='bold')

for ax, (name, props) in zip(axes, bands.items()):
    sig = props['amp'] * np.sin(2 * np.pi * props['freq'] * t)
    sig += np.random.randn(len(t)) * props['amp'] * 0.1  # slight noise
    ax.plot(t, sig, color=props['color'], linewidth=1.2)
    ax.set_ylabel('uV', fontsize=10)
    ax.set_title(f"{name}  |  Typical amplitude: {props['amp']} uV",
                 fontsize=11, color=props['color'], fontweight='bold', loc='left')
    ax.set_ylim(-props['amp']*1.5, props['amp']*1.5)

axes[-1].set_xlabel('Time (seconds)', fontsize=12)
plt.tight_layout()
plt.show()

**Exercise 3:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 4: Frequency Band Power Calculation Using FFT

We can quantify how much power is in each frequency band by computing the FFT and summing the power spectral density within each band's range. This is a fundamental analysis step in clinical EEG.

In [ ]:
# Generate a composite EEG-like signal with known band composition
np.random.seed(7)
fs = 512
duration = 4
t = np.arange(0, duration, 1/fs)

# Composite: strong alpha + moderate theta + weak beta + noise
eeg = (40 * np.sin(2*np.pi*10*t) +    # alpha (dominant)
       25 * np.sin(2*np.pi*6*t) +      # theta
       10 * np.sin(2*np.pi*22*t) +     # beta
       5 * np.sin(2*np.pi*40*t) +      # gamma
       8 * np.random.randn(len(t)))     # noise

# Compute FFT
N = len(eeg)
yf = fft(eeg)
xf = fftfreq(N, 1/fs)[:N//2]
power = 2.0/N * np.abs(yf[:N//2])**2

# Define band ranges
band_ranges = {
    'Delta (1-4 Hz)': (1, 4),
    'Theta (4-8 Hz)': (4, 8),
    'Alpha (8-13 Hz)': (8, 13),
    'Beta (13-30 Hz)': (13, 30),
    'Gamma (30-50 Hz)': (30, 50),
}

# Calculate power in each band
band_powers = {}
for name, (flo, fhi) in band_ranges.items():
    idx = np.where((xf >= flo) & (xf <= fhi))[0]
    band_powers[name] = np.sum(power[idx])

# Plot spectrum with band highlights
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Power spectrum
ax1.semilogy(xf, power, 'k-', linewidth=0.8, alpha=0.7)
colors = ['#2c3e50', '#1abc9c', '#8e44ad', '#e67e22', '#e74c3c']
for (name, (flo, fhi)), c in zip(band_ranges.items(), colors):
    idx = np.where((xf >= flo) & (xf <= fhi))[0]
    ax1.fill_between(xf[idx], power[idx], alpha=0.3, color=c, label=name)
ax1.set_xlabel('Frequency (Hz)', fontsize=12)
ax1.set_ylabel('Power', fontsize=12)
ax1.set_title('Power Spectrum with Band Highlights', fontsize=13, fontweight='bold')
ax1.set_xlim(0, 55)
ax1.legend(fontsize=9)

# Bar chart of band powers
bars = ax2.bar(range(len(band_powers)), list(band_powers.values()), color=colors)
ax2.set_xticks(range(len(band_powers)))
ax2.set_xticklabels([b.split(' ')[0] for b in band_powers.keys()], fontsize=11)
ax2.set_ylabel('Total Power', fontsize=12)
ax2.set_title('Power by Frequency Band', fontsize=13, fontweight='bold')

# Print values
total = sum(band_powers.values())
for name, pwr in band_powers.items():
    print(f"{name}: {pwr:.1f} ({100*pwr/total:.1f}% of total power)")

plt.tight_layout()
plt.show()

**Exercise 4:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 5: Alpha Blocking: The First Clinical Test

One of the first things a neurologist checks during an EEG is **alpha blocking** -- whether the alpha rhythm disappears when the patient opens their eyes.

**Why?** The alpha rhythm is generated by the thalamo-cortical loop when the visual cortex is 'idling.' When the eyes open and visual input floods the cortex, the idle rhythm is disrupted and alpha drops out, replaced by faster beta activity. If alpha doesn't block, something may be wrong.

**Alpha blocking** is one of the most reliable and clinically important EEG phenomena.

In [ ]:
# Simulate alpha blocking: eyes closed vs eyes open
np.random.seed(42)
fs = 512
duration = 3
t = np.arange(0, duration, 1/fs)

# Eyes closed: strong alpha, weak theta, minimal beta
eyes_closed = (40 * np.sin(2*np.pi*10*t) +     # Strong alpha
               12 * np.sin(2*np.pi*6*t) +      # Some theta
               5 * np.sin(2*np.pi*20*t) +      # Minimal beta
               8 * np.random.randn(len(t)))

# Eyes open: suppressed alpha, enhanced beta
eyes_open = (8 * np.sin(2*np.pi*10*t) +        # Suppressed alpha
             8 * np.sin(2*np.pi*6*t) +         # Theta unchanged
             15 * np.sin(2*np.pi*20*t) +       # Enhanced beta
             10 * np.sin(2*np.pi*25*t) +       # More beta
             10 * np.random.randn(len(t)))

# Plot time domain
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('Alpha Blocking Demo: Eyes Closed vs Eyes Open', fontsize=15, fontweight='bold')

# Time domain
axes[0, 0].plot(t[:int(1.5*fs)], eyes_closed[:int(1.5*fs)], color='#8e44ad', linewidth=1.2)
axes[0, 0].set_title('Eyes Closed -- Time Domain', fontsize=12, color='#8e44ad')
axes[0, 0].set_ylabel('uV')
axes[0, 0].set_ylim(-100, 100)

axes[1, 0].plot(t[:int(1.5*fs)], eyes_open[:int(1.5*fs)], color='#e67e22', linewidth=1.2)
axes[1, 0].set_title('Eyes Open -- Time Domain', fontsize=12, color='#e67e22')
axes[1, 0].set_xlabel('Time (s)')
axes[1, 0].set_ylabel('uV')
axes[1, 0].set_ylim(-100, 100)

# Power spectra
for sig, ax, label, color in [
    (eyes_closed, axes[0, 1], 'Eyes Closed', '#8e44ad'),
    (eyes_open, axes[1, 1], 'Eyes Open', '#e67e22')
]:
    N = len(sig)
    yf = fft(sig)
    xf = fftfreq(N, 1/fs)[:N//2]
    psd = 2.0/N * np.abs(yf[:N//2])**2
    ax.plot(xf, psd, color=color, linewidth=1.5)
    # Highlight alpha band
    idx_alpha = np.where((xf >= 8) & (xf <= 13))[0]
    ax.fill_between(xf[idx_alpha], psd[idx_alpha], alpha=0.3, color='#8e44ad')
    ax.set_title(f'{label} -- Power Spectrum', fontsize=12, color=color)
    ax.set_xlim(0, 35)
    ax.set_ylabel('Power')
    ax.annotate('Alpha band\n(8-13 Hz)', xy=(10, max(psd[idx_alpha])*0.8),
                fontsize=9, color='#8e44ad', ha='center')

axes[1, 1].set_xlabel('Frequency (Hz)')
plt.tight_layout()
plt.show()

print('Notice: Alpha power drops dramatically when eyes open.')
print('This is NORMAL. If alpha fails to block, it raises concern for cortical dysfunction.')

**Exercise 5:** Try it yourself!

In [ ]:
# YOUR CODE HERE


## Step 6: Exercise 1: Generate Strong Theta Activity

**Your turn.** Generate a synthetic EEG signal with strong theta activity (4-8 Hz), compute its power spectrum, and identify the dominant frequency band.

**Clinical context:** What clinical state might strong theta represent? Think about: drowsiness, light sleep, temporal lobe pathology in alert adults.

**Exercise 6:** Try it yourself!

In [ ]:
# EXERCISE 1: Strong Theta Signal
# ================================
# 1. Create a 3-second signal at fs=512 Hz dominated by theta (4-8 Hz)
#    Hint: use a 6 Hz sine wave with amplitude ~80 uV
#    Add some smaller alpha and noise for realism
# 2. Plot the time domain signal
# 3. Compute and plot the power spectrum
# 4. Answer: What clinical state might this represent?

# YOUR CODE HERE
fs = 512
duration = 3
t = np.arange(0, duration, 1/fs)

# theta_signal = ???

# Hint: Strong theta could indicate drowsiness/light sleep (normal),
# or temporal lobe dysfunction if the patient is fully awake.

## Step 7: Exercise 2: Frequency Band Power Calculator

**Your turn.** Write a function that takes an EEG signal and sampling rate, and returns the power in each of the five standard EEG frequency bands. Then test it on the composite signal below.

**Exercise 7:** Try it yourself!

In [ ]:
# EXERCISE 2: Band Power Calculator
# ==================================
# 1. Complete the function below
# 2. Test it on the provided signal
# 3. Which band has the most power? Does this match your expectation?

def calculate_band_powers(eeg_signal, fs):
    """
    Calculate the power in each standard EEG frequency band.
    
    Parameters:
        eeg_signal: 1D numpy array of EEG voltage values
        fs: sampling rate in Hz
    
    Returns:
        dict: band name -> power value
    """
    # YOUR CODE HERE
    # Hint: Use fft() and fftfreq() to get the spectrum
    # Then sum the power within each band's frequency range
    pass

# Test signal: heavy delta + some alpha
np.random.seed(99)
fs = 512
t = np.arange(0, 4, 1/fs)
test_signal = (100 * np.sin(2*np.pi*2.5*t) +
               30 * np.sin(2*np.pi*10*t) +
               5 * np.random.randn(len(t)))

# powers = calculate_band_powers(test_signal, fs)
# for band, pwr in powers.items():
#     print(f"{band}: {pwr:.1f}")
# Expected: Delta should dominate, followed by Alpha

## Step 8: Summary: What You See on an EEG Report

When you look at a clinical EEG, the report will describe:

1. **Background activity**: Is the dominant posterior rhythm alpha? What's its frequency and symmetry?
2. **Reactivity**: Does it block with eye opening? Does it change with alertness?
3. **Abnormalities**: Any focal slowing (theta/delta in one region)? Any epileptiform discharges?
4. **Artifacts**: Eye blinks, muscle, electrode, 60 Hz noise (covered in Module 6)

**Key takeaways:**
- The **10-20 System** places electrodes in standardized positions. Letters = brain region, odd = left, even = right, z = midline.
- **Five frequency bands** (delta, theta, alpha, beta, gamma) each carry distinct clinical meaning. Slower = bigger amplitude.
- **Alpha blocking** (disappearance of posterior alpha with eye opening) is a fundamental sign of normal cortical reactivity.

In [ ]:
# Quick reference: EEG frequency bands
print('EEG Frequency Bands Quick Reference')
print('=' * 70)
print(f"{'Band':<12} {'Range':<12} {'Amplitude':<14} {'Significance'}")
print('-' * 70)
ref = [
    ('Delta', '1-4 Hz', '100-200 uV', 'Deep sleep. Pathological if awake.'),
    ('Theta', '4-8 Hz', '50-100 uV', 'Drowsiness/children. Focal = lesion.'),
    ('Alpha', '8-13 Hz', '30-50 uV', 'Relaxed, eyes closed. Blocks with eye opening.'),
    ('Beta', '13-30 Hz', '10-20 uV', 'Alert, anxiety, benzodiazepines.'),
    ('Gamma', '>30 Hz', '5-10 uV', 'Cognitive processing. Muscle artifact risk.'),
]
for band, rng, amp, sig in ref:
    print(f"{band:<12} {rng:<12} {amp:<14} {sig}")

**Exercise 8:** Try it yourself!

In [ ]:
# YOUR CODE HERE
